<a href="https://colab.research.google.com/github/Lexi-cod/Terrain-Identification-and-segmentation/blob/main/DL_KPConv_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KPConv — SemanticKITTI Training Notebook

> **Run cells top-to-bottom.** Each section must complete before the next.

---

## 1. Verify GPU

In [1]:
!nvidia-smi

Thu Mar  5 02:15:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
print('CUDA version:', torch.version.cuda)

CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB
CUDA version: 12.8


---

## 2. Install condacolab

> ⚠️ The runtime will **restart automatically** after this cell. That is expected — just continue from the next cell after restart.

In [3]:
!pip install -q condacolab
import condacolab
condacolab.install()  # Runtime restarts here — continue from Step 3 after restart

✨🍰✨ Everything looks OK!


---

## 3. Create conda environment (Python 3.8)

> Run this after the runtime has restarted from Step 2.

In [4]:
%%bash
conda create -y -n kpconv python=3.8
conda env list

Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local/envs/kpconv

  added / updated specs:
    - python=3.8


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.2.25-hbd8a1cb_0 
  icu                conda-forge/linux-64::icu-78.2-h33c6efd_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_101 
  libffi             conda-forge/linux-64::libffi-3.5.2-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0feb66_18 
  libgcc-ng          conda-forge/linux-64::libgcc-ng-15.2.0-h69a702a_18 
  libgomp            conda-forge/linux-64::libgomp-15.2.0-he0feb66_18 
  liblzma            conda-forge/linux-64::liblzma-5.8.2-hb03c661_0 
  liblzma-devel      conda-forge/linu



==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda




In [5]:
%%bash
/usr/local/envs/kpconv/bin/python --version

Python 3.8.20


---

## 4. Install dependencies into kpconv env

> We install PyTorch 1.10 (required by KPConv-PyTorch) **only** inside the kpconv env.
> We do NOT touch the base Colab environment.

In [6]:
%%bash
# Core scientific packages
/usr/local/envs/kpconv/bin/pip install --quiet \
    numpy==1.21.6 scipy scikit-learn matplotlib pyyaml

# setuptools pin needed for KPConv C++ build
/usr/local/envs/kpconv/bin/pip install --quiet setuptools==59.5.0

# PyTorch 1.10 + CUDA 11.3 (compatible with KPConv-PyTorch)
/usr/local/envs/kpconv/bin/pip install --quiet \
    torch==1.10.0+cu113 torchvision==0.11.1+cu113 \
    --extra-index-url https://download.pytorch.org/whl/cu113

echo 'All dependencies installed.'

All dependencies installed.


---

## 5. Clone KPConv-PyTorch repository

In [7]:
%%bash
rm -rf /content/KPConv-PyTorch
git clone https://github.com/HuguesTHOMAS/KPConv-PyTorch.git /content/KPConv-PyTorch
echo 'Repo contents:'
ls /content/KPConv-PyTorch

Repo contents:
cpp_wrappers
datasets
doc
INSTALL.md
kernels
LICENSE.txt
models
plot_convergence.py
README.md
test_models.py
train_ModelNet40.py
train_NPM3D.py
train_S3DIS.py
train_SemanticKitti.py
train_SensatUrban.py
train_Toronto3D.py
utils
visualize_deformations.py


Cloning into '/content/KPConv-PyTorch'...


---

## 6. Compile C++ wrappers

> This is required for KPConv's radius search and subsampling operations.
> Watch for `SUCCESS` at the end — if it says `FAILED`, check the CUDA version mismatch.

In [8]:
%%bash
cd /content/KPConv-PyTorch/cpp_wrappers
export PATH=/usr/local/envs/kpconv/bin:$PATH

echo '--- Python being used ---'
which python && python --version

echo '--- Compiling wrappers ---'
bash compile_wrappers.sh && echo 'SUCCESS: C++ wrappers compiled.' || echo 'FAILED: Check CUDA/compiler version.'

--- Python being used ---
/usr/local/envs/kpconv/bin/python
Python 3.8.20
--- Compiling wrappers ---
running build_ext
building 'grid_subsampling' extension
Make sure that Python modules winreg, win32api or win32con are installed.
C compiler: gcc -pthread -B /usr/local/envs/kpconv/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /usr/local/envs/kpconv/include -fPIC -O2 -isystem /usr/local/envs/kpconv/include -fPIC

creating build
creating build/temp.linux-x86_64-3.8
creating build/temp.linux-x86_64-3.8/cpp_wrappers
creating build/temp.linux-x86_64-3.8/cpp_wrappers/cpp_utils
creating build/temp.linux-x86_64-3.8/cpp_wrappers/cpp_utils/cloud
creating build/temp.linux-x86_64-3.8/grid_subsampling
compile options: '-I/usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include -I/usr/local/envs/kpconv/include/python3.8 -c'
extra options: '-std=c++11 -D_GLIBCXX_USE_CXX11_ABI=0'
gcc: ../cpp_utils/cloud/cloud.cpp
gcc: grid_subsampling/grid

In file included from /usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarraytypes.h:1969,
                 from /usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/arrayobject.h:4,
                 from wrapper.cpp:2:
/usr/local/envs/kpconv/lib/python3.8/site-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
grid_subsampling/grid_subsampling.cpp: In function ‘void grid_subsampling(std::vector<PointXYZ>&, std::vector<PointXYZ>&, std::vector<float>&, std::vector<float>&, std::vector<int>&, std::vector<int>&, float, int)’:
grid_subsampling/grid_subsampling.cpp:99:39: warning: comparison of integer expr

---

## 7. Mount Google Drive and link SemanticKITTI dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
%%bash
DATA_DIR=/content/KPConv-PyTorch/Data/SemanticKitti
SRC=/content/drive/MyDrive/Data/semantickitti_subsampled

mkdir -p $DATA_DIR/sequences

# Link train, val, and test sequences
ln -sf $SRC/sequences/* $DATA_DIR/sequences/

# Link YAML config
ln -sf $SRC/semantic-kitti.yaml $DATA_DIR/semantic-kitti.yaml

echo 'Sequences linked:'
ls $DATA_DIR/sequences/

Sequences linked:
00
01
02
03
04
05
06
07
08
09
10
11
12
13
14
15
16
17
18
19
20
21


---

## 8. Patch config for Colab GPU memory

> The default batch sizes in KPConv are tuned for multi-GPU workstations.
> This cell reduces them so training fits in Colab GPU VRAM (16–40 GB).

In [20]:
import re

config_path = '/content/KPConv-PyTorch/train_SemanticKitti.py'

with open(config_path, 'r') as f:
    src = f.read()

# Reduce batch sizes for single-GPU Colab
src = re.sub(r'self\.batch_num\s*=\s*\d+', 'self.batch_num = 6', src)
src = re.sub(r'self\.val_batch_num\s*=\s*\d+', 'self.val_batch_num = 4', src)

# Point checkpoint saving to Google Drive so progress is kept if Colab disconnects
src = re.sub(
    r"(self\.saving_path\s*=\s*)'[^']*'",
    r"\1'/content/drive/MyDrive/DL_data/kpconv_checkpoints'",
    src
)

with open(config_path, 'w') as f:
    f.write(src)

print('Config patched:')
for line in src.split('\n'):
    if any(k in line for k in ['batch_num', 'val_batch_num', 'saving_path']):
        print(' ', line.strip())

Config patched:
  batch_num = 8
  val_batch_num = 8
  saving_path = None
  config.saving_path = None
  config.saving_path = sys.argv[1]


---

## 9. Verify dataset import

In [21]:
import os, sys

project_root = '/content/KPConv-PyTorch'
os.chdir(project_root)
sys.path.insert(0, os.path.join(project_root, 'datasets'))

# Ensure __init__.py exists
open(os.path.join(project_root, 'datasets', '__init__.py'), 'a').close()

print('Project root contents:', os.listdir(project_root))
print('Datasets folder:', os.listdir(os.path.join(project_root, 'datasets')))

Project root contents: ['kernels', 'train_S3DIS.py', '.git', 'test_models.py', 'cpp_wrappers', 'plot_convergence.py', 'train_ModelNet40.py', 'models', 'utils', 'visualize_deformations.py', 'train_SensatUrban.py', 'doc', 'train_SemanticKitti.py', 'train_Toronto3D.py', 'datasets', 'INSTALL.md', 'README.md', 'LICENSE.txt', 'train_NPM3D.py', 'Data', '.gitignore', 'results']
Datasets folder: ['Toronto3D.py', 'SensatUrban.py', '__init__.py', '__pycache__', 'SemanticKitti.py', 'NPM3D.py', 'ModelNet40.py', 'common.py', 'S3DIS.py']


---

## 10. Train KPConv on SemanticKITTI

> ⚠️ Make sure all cells above have run successfully before starting training.
> Checkpoints will be saved to your Google Drive automatically.

In [22]:
import os
os.environ['MPLBACKEND'] = 'Agg'  # Non-interactive backend (no display needed)

%cd /content/KPConv-PyTorch
!/usr/local/envs/kpconv/bin/python train_SemanticKitti.py

Streaming output truncated to the last 5000 lines.
2068209.0 446591.0      0.0   7060.0  33459.0  15530.0   7407.0   3451.0    652.0  16567.0    409.0  38155.0    459.0 603350.0 734194.0 725711.0  53659.0  30430.0  42569.0   4072.0 
473880.0 5332513.0      0.0  15727.0 213580.0  61498.0   1728.0    379.0   1041.0  13388.0  14261.0  14496.0    246.0  29490.0  97177.0 230635.0   1398.0   3346.0    373.0     38.0 
  4593.0   1275.0      0.0   7544.0    235.0     19.0    589.0    219.0    844.0    172.0      8.0    934.0      0.0   1938.0   3143.0  26884.0      1.0    756.0     28.0      0.0 
  4467.0   5592.0      0.0  23324.0    271.0    953.0    994.0   3284.0    347.0    210.0      1.0    196.0     57.0    146.0   4299.0  33748.0      0.0    422.0      7.0      0.0 
 12471.0  11870.0      0.0     48.0  71330.0   1428.0      0.0      6.0     10.0    276.0     24.0    177.0      0.0  19182.0    240.0  19382.0    189.0      3.0      0.0      0.0 
 35636.0  75654.0      0.0   4691.0 109216

---

## 11. Optional diagnostics

Run these any time to check GPU usage or disk space.

In [23]:
# GPU memory and utilization
!nvidia-smi

# Disk space
!df -h /content

Thu Mar  5 05:48:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             46W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [24]:
# List saved checkpoints on Drive
import os
ckpt_dir = '/content/drive/MyDrive/DL_data/kpconv_checkpoints'
if os.path.exists(ckpt_dir):
    for f in sorted(os.listdir(ckpt_dir)):
        print(f)
else:
    print('No checkpoints directory yet — training may not have started.')

No checkpoints directory yet — training may not have started.
